# Smart Hospital - Train YOLO 3 Class No Idle

Dataset nay KHONG train bbox cho Binh_Thuong. Binh_Thuong la trang thai idle cua app khi YOLO khong detect duoc gesture nao.

1. Runtime -> Change runtime type -> T4 GPU -> Save
2. Upload `dataset_3class.zip` len Drive: `MyDrive/SmartHospital/dataset_3class.zip`
3. Chay lan luot tung cell ben duoi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!rm -rf /content/dataset /content/dataset_3class.zip
!cp "/content/drive/MyDrive/SmartHospital/dataset_3class.zip" /content/dataset_3class.zip
!unzip -q /content/dataset_3class.zip -d /content/
!echo "--- data.yaml ---"
!cat /content/dataset/data.yaml
!echo "--- counts ---"
!find /content/dataset/images/train -type f | wc -l
!find /content/dataset/images/val -type f | wc -l

In [ ]:
!pip install ultralytics -q
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
from collections import Counter

label_counts = Counter()
empty_labels = 0
for split in ['train', 'val']:
    for label_path in Path('/content/dataset/labels').joinpath(split).glob('*.txt'):
        text = label_path.read_text().strip()
        if not text:
            empty_labels += 1
            continue
        for line in text.splitlines():
            label_counts[int(float(line.split()[0]))] += 1

print('Class boxes:', dict(label_counts))
print('Negative/background images:', empty_labels)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=120,
    imgsz=640,
    batch=16,
    device=0,
    patience=30,
    project='/content/runs',
    name='gesture_3class_no_idle',
    cos_lr=True,
    close_mosaic=10,
)

In [ ]:
from ultralytics import YOLO

best_path = '/content/runs/gesture_3class_no_idle/weights/best.pt'
model = YOLO(best_path)
metrics = model.val(data='/content/dataset/data.yaml', imgsz=640, device=0)
print(metrics)

In [ ]:
from pathlib import Path
from ultralytics import YOLO

model = YOLO('/content/runs/gesture_3class_no_idle/weights/best.pt')
sample_images = list(Path('/content/dataset/images/val').glob('*'))[:12]
model.predict(sample_images, imgsz=640, conf=0.25, save=True)
print('Saved predictions under:', model.predictor.save_dir)

In [ ]:
!mkdir -p "/content/drive/MyDrive/SmartHospital"
!cp /content/runs/gesture_3class_no_idle/weights/best.pt "/content/drive/MyDrive/SmartHospital/best_3class.pt"
!cp /content/runs/gesture_3class_no_idle/weights/best.pt "/content/drive/MyDrive/SmartHospital/best.pt"
print('Saved: MyDrive/SmartHospital/best_3class.pt')
print('Also replaced: MyDrive/SmartHospital/best.pt')

## Sau khi train xong

Tai `best_3class.pt` hoac `best.pt` tu Drive ve may, dat vao `D:\\DEV_AI\\best.pt`, roi bao Codex copy vao `models/best.pt` va chay app.

Ky vong cua model moi: khong con box `Binh_Thuong`; khi khong co gesture hanh dong, app se hien `Stable: Binh_Thuong` do logic idle.